In [1]:
# load modules
import geopandas as gpd
import pandas as pd
import fiona
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

import matplotlib.pyplot as plt

In [2]:
# load the file with the narratives from the crash report

narratives1 = pd.read_csv("Data/Franklin_DocumentNumber_Narrative_part1.csv")
narratives2 = pd.read_csv("Data/Franklin_DocumentNumber_Narrative_part2.csv")

In [ ]:
# merge the files in one
narratives = pd.concat([narratives1, narratives2], ignore_index=True)

In [ ]:
# save the narratives as its own csv
narratives.to_csv("Data/Franklin_DocumentNumber_Narrative_full.csv", index=False)

In [24]:
narratives

,DocumentNumber,County,Narrative
0,20202168971,Franklin County,Unit# 1 a pedestrian was crossing US-40 from s...
1,20202168999,Franklin County,Unit 1 was westbound on SR 317 towards Parsons...
2,20202172825,Franklin County,Unit 2 was traveling south on SR 104 south of ...
3,20202183697,Franklin County,Unit 1 was crossing US 40 outside of the cross...
4,20202206898,Franklin County,Unit 1 was driving south on SR 3 just north of...
...,...,...,...
115676,20258172808,Franklin County,On Sunday; September 21; 2025; Unit 2 was stop...
115677,20258173076,Franklin County,On above date and time U-1 failed to yield at ...
115678,20258173087,Franklin County,Unit #1 was traveling eastbound on Josephus Ln...
115679,20258173091,Franklin County,During the night of September 13th; 2025 after...


In [5]:
# now search for the top 10 (or top 20) fast food chains in the narratives

# load the fastfood places
fastfood = gpd.read_file("Data/fastfood_all_ohio.gpkg", layer="fastfood")

# check the occurances of the pizza chains
fastfood_counts = fastfood["name"].value_counts()

# convert to a DataFrame for nicer display
fastfood_counts_df = fastfood_counts.reset_index()

print(fastfood_counts[:10])
top10_fastfood = fastfood_counts[:10]
top20_fastfood = fastfood_counts[:20]


# load the pizza places
pizza = gpd.read_file("Data/pizza_places_ohio.gpkg", layer="pizza_places")
# check the occurances of the pizza chains
pizza_counts = pizza["name"].value_counts()

# convert to dataframe for nicer display
pizza_counts_df = pizza_counts.reset_index()
#pizza_counts_df.columns = ["fastfood_chain", "count"]

print(pizza_counts[:10])
top10_pizza = pizza_counts[:10]
top20_pizza = pizza_counts[:20]

name
McDonald's     614
Subway         585
Wendy's        368
Taco Bell      325
Burger King    281
Arby's         245
Chipotle       214
Domino's       162
Dunkin'        162
KFC            156
Name: count, dtype: int64
name
Pizza Hut                174
Domino's                 161
Little Caesars            99
Papa John's               83
Marco's Pizza             83
Donatos Pizza             69
LaRosa's Pizzeria         30
Jet's Pizza               19
Gionino's Pizzeria        18
East of Chicago Pizza     18
Name: count, dtype: int64


In [6]:
pizza_10names = top10_pizza.index[:]
fastfood_10names = top10_fastfood.index[:]

print(pizza_10names)
print(fastfood_10names)

Index(['Pizza Hut', 'Domino's', 'Little Caesars', 'Papa John's',
       'Marco's Pizza', 'Donatos Pizza', 'LaRosa's Pizzeria', 'Jet's Pizza',
       'Gionino's Pizzeria', 'East of Chicago Pizza'],
      dtype='object', name='name')
Index(['McDonald's', 'Subway', 'Wendy's', 'Taco Bell', 'Burger King', 'Arby's',
       'Chipotle', 'Domino's', 'Dunkin'', 'KFC'],
      dtype='object', name='name')


In [7]:
# the spelling of the restaurant places can vary in the reports, so I will look for apostrophes in the name and 
# just go with the name before the apostrophe
# (the order matters)
pizza_names_stub = [name.split("s ")[0] for name in top10_pizza.index[:]]
pizza_names_stub = [name.split("'")[0] for name in top10_pizza.index[:]]
print(pizza_names_stub)


['Pizza Hut', 'Domino', 'Little Caesars', 'Papa John', 'Marco', 'Donatos Pizza', 'LaRosa', 'Jet', 'Gionino', 'East of Chicago Pizza']


In [8]:
# the spelling of the restaurant places can vary in the reports, so I will look for apostrophes in the name and 
# just go with the name before the apostrophe
# (the order matters)

# same for the fastfood places
fastfood_names_stub = [name.split("'")[0] for name in top10_fastfood.index[:]]
print(fastfood_names_stub)

['McDonald', 'Subway', 'Wendy', 'Taco Bell', 'Burger King', 'Arby', 'Chipotle', 'Domino', 'Dunkin', 'KFC']


In [9]:
# now we have the list of the top 10 restaurant names,
# now look through the narratives and search for these words
# issue might be that there is different writing of the restaurants in the narrative
# like Dominos instead of Domino's

# set up a new dictionary to store the information
narrative_names = {}

for word in fastfood_names_stub:
    if word.strip() == "Arby":
        mask = narratives['Narrative'].str.contains(r'\sArby', case=False, na=False)
    else:
        mask = narratives['Narrative'].str.contains(word, case=False, na=False)
    print(word)
    matching_reports = narratives.loc[mask, 'DocumentNumber']
    narrative_names[word] = matching_reports


McDonald
Subway
Wendy
Taco Bell
Burger King
Arby
Chipotle
Domino
Dunkin
KFC


In [10]:
# check all the DocumentNumbers for a certain fast food chain
print(narrative_names["McDonald"])


497       20203168201
570       20203169023
714       20203171089
986       20203175958
1513      20203183836
             ...     
113202    20253155992
113591    20253160667
113951    20253164925
114296    20253168048
115618    20258158560
Name: DocumentNumber, Length: 313, dtype: int64


In [11]:
# just a check to see how often a certain chain is mentioned
key_lengths = {key: len(narrative_names[key]) for key in narrative_names}

print(key_lengths)

{'McDonald': 313, 'Subway': 21, 'Wendy': 90, 'Taco Bell': 49, 'Burger King': 22, 'Arby': 8, 'Chipotle': 22, 'Domino': 6, 'Dunkin': 9, 'KFC': 23}


In [12]:
print(len(narrative_names["McDonald"]))
print(narrative_names["McDonald"])
sorted_narrative_names = sorted(narrative_names, key=lambda k: len(narrative_names[k]), reverse=True)
print(sorted_narrative_names)


313
497       20203168201
570       20203169023
714       20203171089
986       20203175958
1513      20203183836
             ...     
113202    20253155992
113591    20253160667
113951    20253164925
114296    20253168048
115618    20258158560
Name: DocumentNumber, Length: 313, dtype: int64
['McDonald', 'Wendy', 'Taco Bell', 'KFC', 'Burger King', 'Chipotle', 'Subway', 'Dunkin', 'Arby', 'Domino']


In [13]:
# just a nicer display
for key in sorted_narrative_names:
    value = narrative_names[key]
    print(f"{key}: {len(value)} mentions")


McDonald: 313 mentions
Wendy: 90 mentions
Taco Bell: 49 mentions
KFC: 23 mentions
Burger King: 22 mentions
Chipotle: 22 mentions
Subway: 21 mentions
Dunkin: 9 mentions
Arby: 8 mentions
Domino: 6 mentions


In [14]:
print(narrative_names["McDonald"])

497       20203168201
570       20203169023
714       20203171089
986       20203175958
1513      20203183836
             ...     
113202    20253155992
113591    20253160667
113951    20253164925
114296    20253168048
115618    20258158560
Name: DocumentNumber, Length: 313, dtype: int64


In [15]:
print(narrative_names["Arby"])

29429     20223005258
30842     20223026315
34391     20223079568
87655     20243140054
96775     20243245802
99196     20248243505
104756    20253058957
106800    20253082041
Name: DocumentNumber, dtype: int64


We need to identify now if the mention of McDonald's has anything to do with the crash or is just a location marker

So print the narrative.

In [16]:
# convert dict to df:
narrative_names_df = pd.DataFrame(narrative_names)
print(narrative_names_df)


#narrative_names["McDonald\'s"][]

            McDonald  Subway  Wendy  Taco Bell   Burger King  Arby  Chipotle  \
497     2.020317e+10     NaN    NaN        NaN           NaN   NaN       NaN   
570     2.020317e+10     NaN    NaN        NaN           NaN   NaN       NaN   
714     2.020317e+10     NaN    NaN        NaN           NaN   NaN       NaN   
986     2.020318e+10     NaN    NaN        NaN           NaN   NaN       NaN   
1513    2.020318e+10     NaN    NaN        NaN           NaN   NaN       NaN   
...              ...     ...    ...        ...           ...   ...       ...   
113951  2.025316e+10     NaN    NaN        NaN           NaN   NaN       NaN   
114296  2.025317e+10     NaN    NaN        NaN           NaN   NaN       NaN   
115281           NaN     NaN    NaN        NaN  2.025805e+10   NaN       NaN   
115292           NaN     NaN    NaN        NaN           NaN   NaN       NaN   
115618  2.025816e+10     NaN    NaN        NaN           NaN   NaN       NaN   

              Domino  Dunkin  KFC  
497

From this I learn that there are in total 557 mentions of the top 10 fastfood chains. After correcting that "nearby" conatins the word "Arby", the list seems to make sense.

In [17]:
# print a random narrative for checking:

print(narrative_names["McDonald"])
mcd_document_numbers = narrative_names["McDonald"]


497       20203168201
570       20203169023
714       20203171089
986       20203175958
1513      20203183836
             ...     
113202    20253155992
113591    20253160667
113951    20253164925
114296    20253168048
115618    20258158560
Name: DocumentNumber, Length: 313, dtype: int64


In [18]:
# I want to merge all fastfood documentnumbers in one file, because we don't care which chain it is
fastfood_document_numbers = [number for sublist in narrative_names.values() for number in sublist]
print(fastfood_document_numbers)
print("")
print(len(fastfood_document_numbers))

[20203168201, 20203169023, 20203171089, 20203175958, 20203183836, 20203187027, 20203203074, 20203205393, 20203206891, 20203209872, 20203213688, 20203219266, 20203220156, 20203223118, 20203223786, 20203230214, 20213003067, 20213007603, 20213008111, 20213019000, 20213032957, 20213036003, 20213036618, 20213042366, 20213064942, 20213076893, 20213079954, 20213083612, 20213086224, 20213092935, 20213098667, 20213100942, 20213110226, 20213110998, 20213113543, 20213124404, 20213126391, 20213126401, 20213131830, 20213133761, 20213133929, 20213137368, 20213137369, 20213138405, 20213147182, 20213147903, 20213158332, 20213167027, 20213167993, 20213172526, 20213172991, 20213178281, 20213181554, 20213184797, 20213196046, 20213202678, 20213211246, 20213218825, 20213236606, 20213247289, 20213252304, 20214153745, 20214172355, 20223009625, 20223017802, 20223018939, 20223024456, 20223030614, 20223035880, 20223041337, 20223044905, 20223048055, 20223054870, 20223057259, 20223059341, 20223072085, 20223073333

In [19]:
# now check the narrative file for these entries
fastfood_filtered_narratives = narratives[narratives['DocumentNumber'].isin(fastfood_document_numbers)]

In [20]:
fastfood_filtered_narratives

,DocumentNumber,County,Narrative
497,20203168201,Franklin County,Unit 1 was traveling south on Main Street pass...
570,20203169023,Franklin County,On 10-6-2020 Unit #1 was traveling from North ...
714,20203171089,Franklin County,On 10/9/2020 at approximately 8:55 PM; Officer...
986,20203175958,Franklin County,Diver of Unit 2 states she was in the drive th...
1513,20203183836,Franklin County,On 10/23/2020 at approxiamtely 4:53 PM; Office...
...,...,...,...
113951,20253164925,Franklin County,Driver of Unit #1 was driving through the park...
114296,20253168048,Franklin County,On 9/11/2025 the driver of Unit 2 stated he wa...
115281,20258051287,Franklin County,On March 19; 2025; at around 1613 hours; unit ...
115292,20258054115,Franklin County,Unit #1 was westbound on E. Broad Street when ...


In [21]:
fastfood_filtered_narratives.loc[1513,"Narrative"]

# haha, in this exact entry, McDonald is the name of a person and has nothing to do with the location of a fastfood restaurant

'On 10/23/2020 at approxiamtely 4:53 PM; Officer A. McDonald #2811 and Officer S. Weeks #2832; wearing the uniform of the day and working CPD marked cruiser R-9160 as unit 122 were dispatched to I-70 W/B at S. Ohio Ave. in response to an auto accident. Upon arrival Officer McDonald spoke with the daughter of the driver of unit #2 Kalpana; who arrived at scene after the accident occured. Due to the language barrier Kalpana acted as a interpreter for officers and Guru Timsina; the operator of unit #1. According to Mr. Timsina he was struck in the rear of his vehicle by unit #1. Officer Weeks spoke to the driver of unit #1 who was identified by her Ohio drivers license as Mercedes Martell. Ms. Martell stated that unit #2 came to a abrupt stop and she crashed into the rear of unit #2. Both parties were issued a incident report number and advised to contact their insurance companies. '

In [22]:
fastfood_filtered_narratives.loc[115292,"Narrative"]

# here, "domino" is used in "domino effect" and not as the fastfood chain

'Unit #1 was westbound on E. Broad Street when he failed to come to a complete stop for the traffic ahead of him; causing him to crash into the rear of Unit #2 causing a domino effect with Unit #2 to crash into Unit #3 and Unit #3 to crash into Unit #4.'

In [23]:
# save the fastfood_filtered_narratives file as its own csv to go back to this
fastfood_filtered_narratives.to_csv("Data/Fastfood_filtered_narratives_DocumentNumber.csv", index=False)

As some accidents might not even be close to a fast food chain, we don't need to consider them at a later stage.

Now we want to match the locations of the accidents with the maps of the fast-food drive-throughs.